# 09 — Real Strength Evidence Protocol

## 1–3. Frozen protocol, protocol hash, and data origin

This notebook validates the pre-registered Strength V1 protocol. It never tunes the target, features, baselines, models, or thresholds. Real and synthetic origins are never combined.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import matplotlib.pyplot as plt
from metaform_ds.config import DataScienceConfig
from metaform_ds.modeling.strength.evidence import unavailable_strength_evidence_result
from metaform_ds.modeling.strength.evidence_reports import render_strength_real_evidence_report
from metaform_ds.modeling.strength.protocol import FrozenStrengthEvidenceProtocol
from metaform_ds.strength_evidence import StrengthEvidenceExecutionError, run_real_strength_evidence, run_synthetic_strength_protocol_validation
settings = DataScienceConfig.from_env()
protocol = FrozenStrengthEvidenceProtocol()
print(f'Protocol version: {protocol.protocol_version}')
print(f'Protocol hash: {protocol.protocol_hash}')
print(protocol.canonical_json())

In [ ]:
try:
    real_result = run_real_strength_evidence(settings, protocol)
except StrengthEvidenceExecutionError as error:
    print(f'REAL evaluation failed safely: {error}')
    real_result = unavailable_strength_evidence_result(protocol)
print(f'REAL DATA STATUS: {real_result.data_origin}; evidence: {real_result.evidence_state}')
if real_result.data_origin == 'UNAVAILABLE': print('REAL strength evaluation pending. No synthetic fallback occurred.')

## 4–8. Quality, eligibility, evidence scope, source coverage, and sufficiency

All displays below are aggregate. Personal temporal validation is explicitly distinct from multi-user generalization.

In [ ]:
print('QUALITY SUMMARY')
print(asdict(real_result.quality))
print('ELIGIBILITY AND SOURCE COVERAGE')
print(asdict(real_result.eligibility))
print(f'EVIDENCE SCOPE: {real_result.evidence_scope}')
print(f'REASON CODES: {real_result.reason_codes or ("NONE",)}')

## 9–14. Frozen comparison, temporal stability, source/history/cadence segments, and direction

MAE remains primary. Direction accuracy is descriptive only. Sparse segments remain insufficient rather than being generalized.

In [ ]:
evaluation = real_result.evaluation
if evaluation is None:
    print('Frozen four-method REAL comparison pending sufficient eligible data.')
else:
    print(evaluation.comparison.to_string(index=False))
    print(evaluation.stability.to_string(index=False))
    print(evaluation.source_type_metrics.to_string(index=False))
    print(evaluation.history_metrics.to_string(index=False))
    print(evaluation.cadence_metrics.to_string(index=False))

In [ ]:
if evaluation is not None and not evaluation.comparison.empty:
    view = evaluation.comparison.sort_values('mae_kg')
    fig, ax = plt.subplots()
    ax.bar(view['method'], view['mae_kg'])
    ax.set(title='REAL temporal MAE by frozen method', xlabel='Method', ylabel='MAE (kg)')
    ax.tick_params(axis='x', rotation=20)
    plt.show()
else: print('REAL MAE chart pending sufficient data.')

In [ ]:
if evaluation is not None and not evaluation.fold_metrics.empty:
    fig, ax = plt.subplots()
    for method, rows in evaluation.fold_metrics.groupby('method'):
        ax.plot(rows['fold_id'], rows['mae_kg'], marker='o', label=method)
    ax.set(title='REAL fold MAE by frozen method', xlabel='Temporal fold', ylabel='MAE (kg)')
    ax.tick_params(axis='x', rotation=90)
    ax.legend()
    plt.show()
else: print('REAL fold chart pending sufficient data.')

## Separate synthetic mechanics validation

The following run is explicitly SYNTHETIC. It validates deterministic protocol mechanics only and contributes nothing to REAL evidence or the integration gate.

In [ ]:
synthetic_result = run_synthetic_strength_protocol_validation(settings, protocol)
print(f'SYNTHETIC MECHANICS ONLY: {synthetic_result.evidence_state}; predictions: {synthetic_result.aggregate_summary.prediction_count}')
print(f'Integration review allowed: {synthetic_result.integration_review_allowed}')

## 15–17. Integration-review gate, limitations, and next decision gate

The gate can permit only a future integration review, never production readiness. If evidence is insufficient, accumulate additional read-only history under this unchanged protocol.

In [ ]:
print(f'Integration review allowed: {real_result.integration_review_allowed}')
print(f'Preferred method: {real_result.aggregate_summary.preferred_method or "Not evaluated"}')
print('Limitations:', *real_result.limitations, sep='\n- ')
report = render_strength_real_evidence_report(real_result)
artifact_dir = settings.artifact_dir
if not artifact_dir.is_absolute() and Path.cwd().name == 'notebooks': artifact_dir = Path('..') / artifact_dir
artifact_dir.mkdir(parents=True, exist_ok=True)
report_path = artifact_dir / 'strength_real_evidence_v1.md'
report_path.write_text(report, encoding='utf-8')
print(f'Ignored aggregate report: {report_path}')
print(report)